# Visual ML Foundations: Out-of-Fold Cross Validation (GroupKFold)
### Preventing Data Leakage Across Patient Studies and Series

---

## 1. Concept Overview
The most common mistake on Kaggle is **Data Leakage**:
* In MRI imaging, one patient (`StudyInstanceUID`) may have 3 series (Sagittal, Coronal, Axial).
* If you perform random train/test splitting, slices from the same patient land in both train and validation sets!
* The model memorizes patient-specific bone anatomy, showing an impressive $0.99$ validation score, but collapses on the test leaderboard.
* **The Solution**: **GroupKFold** grouped strictly on `StudyInstanceUID`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold

# Simulate 20 patient studies, each with 3 series (60 total scans)
n_patients = 20
studies = np.repeat(np.arange(n_patients), 3)
scans = np.arange(len(studies))

gkf = GroupKFold(n_splits=5)
fold_assignments = np.zeros(len(studies))

for fold, (train_idx, val_idx) in enumerate(gkf.split(scans, groups=studies)):
    fold_assignments[val_idx] = fold

# Visualizing fold distributions
fig, ax = plt.subplots(figsize=(14, 4))
scatter = ax.scatter(scans, studies, c=fold_assignments, cmap='tab10', s=120, edgecolors='black')
ax.set_title("GroupKFold Distribution: Zero Patient Leakage Across 5 Folds", fontsize=12, fontweight='bold')
ax.set_xlabel("Scan Index (0 to 59)")
ax.set_ylabel("Patient Study UID (0 to 19)")
ax.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=ax, ticks=range(5))
cbar.set_label("Assigned Validation Fold", fontweight='bold')

plt.tight_layout()
plt.show()
print("Patient group partitioning verified: No patient crosses fold boundaries.")
